---
### **Domain: Global Venomous Snake Ecology Assistant**

**Chatbot Name: VenomWise**

*"VenomWise is a domain-specific RAG chatbot designed to educate users on venomous snake species found across different biogeographic regions of the world, emphasizing their taxonomic classification, venom characteristics, and ecological function as apex/mesopredators within their respective ecosystems. The system aims to counter common public misconceptions about venomous snakes by grounding all responses strictly in curated, verifiable source material."*

---





---
### **Domain Data Collection from Wikipedia**
**Purpose:**
Retrieves real-world educational data about venomous snakes from Wikipedia using
the MediaWiki Action API. The cell downloads plain-text articles for five venomous
snake species from different world regions and saves them as `.txt` files inside
the `./my_data/` folder. These files become the knowledge base for the RAG chatbot.

**Input:**
- Internet connection to Wikipedia
- `requests` and `os` Python libraries
- A list of target Wikipedia article titles
- No API key required because Wikipedia's API is public

**Output:**
- A `./my_data/` folder created in the notebook runtime
- Five text files containing Wikipedia article content:
  - `black_mamba.txt`
  - `king_cobra.txt`
  - `inland_taipan.txt`
  - `fer_de_lance.txt`
  - `saw_scaled_viper.txt`
- Source attribution, URL, and license information stored at the top of every
  saved file
- Success or error messages displayed for each snake article

---
**Line-by-line breakdown:**

- `import requests` — Imports the `requests` library, which allows Python to
send HTTP requests to websites and APIs. It is used to communicate with the
Wikipedia API and retrieve article content.

- `import os` — Imports Python's operating-system library. It is used to create
the `my_data` folder where the downloaded snake information will be saved.

- `os.makedirs("./my_data", exist_ok=True)` — Creates a directory called
`my_data` in the current notebook location. The `exist_ok=True` option prevents
an error if the folder already exists, allowing the cell to be safely run again.

- `HEADERS = {...}` — Creates a dictionary containing HTTP request headers.

- `"User-Agent": "ITCC508-RAG-Lab/1.0 (...)"` — Identifies the notebook as an
educational project when sending requests to Wikipedia. Wikipedia requires
automated tools to provide a descriptive User-Agent header; otherwise, requests
may be blocked or return invalid responses. Replace `your_email@example.com`
with your actual email address before submitting the project.

- `snake_pages = {...}` — Creates a Python dictionary containing the target
venomous snake articles.

- `"black_mamba": "Black mamba"` — The dictionary key (`black_mamba`) becomes
the saved file name, while the value (`Black mamba`) is the Wikipedia article
title to retrieve.

- `"king_cobra": "King cobra"` — Retrieves information about the King Cobra,
representing Asia.

- `"inland_taipan": "Inland taipan"` — Retrieves information about the Inland
Taipan, representing Australia.

- `"fer_de_lance": "Bothrops asper"` — Retrieves information about the
Fer-de-lance snake, which is listed on Wikipedia under its scientific name,
*Bothrops asper*, and represents Central and South America.

- `"saw_scaled_viper": "Echis carinatus"` — Retrieves information about the
Saw-scaled Viper, listed on Wikipedia under its scientific name,
*Echis carinatus*.

- `def get_wikipedia_content(title):` — Defines a reusable function that
accepts a Wikipedia article title and returns the article's resolved title and
plain-text content.

- `url = "https://en.wikipedia.org/w/api.php"` — Defines the official English
Wikipedia MediaWiki Action API endpoint. Using an API is more reliable than
copying text manually or scraping webpage HTML.

- `params = {...}` — Stores the API request parameters in a dictionary.

- `"action": "query"` — Tells Wikipedia that the request is asking to retrieve
information.

- `"prop": "extracts"` — Requests the main article text, called an extract.

- `"explaintext": True` — Requests plain text instead of HTML. This is important
for RAG because HTML tags and webpage formatting would add unnecessary noise to
the document chunks and embeddings.

- `"format": "json"` — Requests the response in JSON format, which Python can
easily convert into dictionaries.

- `"titles": title` — Uses the title passed into the function, such as
`"Black mamba"` or `"King cobra"`.

- `"redirects": 1` — Allows Wikipedia to automatically follow redirects. This
is useful when common names redirect to scientific names or differently titled
articles.

- `response = requests.get(...)` — Sends an HTTP GET request to Wikipedia using
the API URL, request parameters, User-Agent header, and a 10-second timeout.

- `params=params` — Automatically converts the parameters dictionary into a
proper URL query string.

- `headers=HEADERS` — Attaches the User-Agent header so Wikipedia can identify
the educational tool making the request.

- `timeout=10` — Stops the request if Wikipedia does not respond within
10 seconds, preventing the notebook from waiting indefinitely.

- `response.raise_for_status()` — Checks whether the request was successful.
If Wikipedia returns an error code, such as 403, 404, or 500, Python raises an
error that can be handled by the `try-except` block.

- `data = response.json()` — Converts the JSON response returned by Wikipedia
into a Python dictionary.

- `pages = data["query"]["pages"]` — Navigates through the nested API response
to access the dictionary containing the requested Wikipedia page.

- `for page_id, page_data in pages.items():` — Loops through the returned page
data. Wikipedia stores pages using a page ID, which is not known before making
the request.

- `if page_id == "-1" or "extract" not in page_data:` — Checks whether the
requested page does not exist or whether no article text was returned.

- `return None, None` — Returns empty values when the page cannot be found or
does not contain usable text.

- `return page_data.get("title"), page_data.get("extract")` — Returns the
resolved article title and its complete plain-text extract when retrieval is
successful.

- `for filename, page_title in snake_pages.items():` — Loops through all snake
entries in the dictionary. Each loop uses `filename` for the output file name
and `page_title` for the Wikipedia request.

- `try:` — Starts error handling for each snake article. This ensures that if
one article fails to download, the remaining articles can still be processed.

- `resolved_title, content = get_wikipedia_content(page_title)` — Calls the
retrieval function and stores the returned Wikipedia title and article text.

- `if content is None or len(content.strip()) == 0:` — Checks whether the
retrieved content is missing or contains only blank spaces.

- `print(f"❌ No content found for: {page_title}")` — Displays an error message
when Wikipedia does not return usable article content.

- `continue` — Skips the current snake and proceeds to the next article instead
of creating an empty text file.

- `filepath = f"./my_data/{filename}.txt"` — Creates the output file path using
an f-string. For example, the Black Mamba article is saved as
`./my_data/black_mamba.txt`.

- `with open(filepath, "w", encoding="utf-8") as f:` — Opens the output file in
write mode. The `with` statement automatically closes the file after writing.
`encoding="utf-8"` supports scientific names, special symbols, accented
characters, and other non-English characters found in Wikipedia content.

- `f.write(f"Source: Wikipedia — {resolved_title}\n")` — Writes the Wikipedia
article title into the file as source attribution.

- `f.write(f"URL: ...")` — Writes the direct Wikipedia URL into the file so the
source can be verified later.

- `resolved_title.replace(' ', '_')` — Replaces spaces with underscores because
Wikipedia article URLs use underscores, such as `Black_mamba`.

- `f.write("Licensed under CC BY-SA 4.0 ...")` — Adds a license notice.
Wikipedia content is distributed under the Creative Commons Attribution-
ShareAlike 4.0 license, so the source must be acknowledged.

- `f.write("=" * 70 + "\n\n")` — Creates a horizontal separator line between
the source metadata and the actual article text.

- `f.write(content)` — Writes the retrieved Wikipedia article content into the
text file.

- `print(f"✅ Saved: {filepath} ({len(content)} characters)")` — Displays a
success message containing the saved file path and the number of characters
downloaded. This provides a quick check that the article was retrieved
successfully.

- `except Exception as e:` — Catches unexpected errors, such as connection
problems, timeouts, blocked requests, or invalid API responses.

- `print(f"❌ Failed for {page_title}: {e}")` — Displays the snake article that
failed and the specific error message, then allows the loop to continue with
the next snake.

---

### **Why this step is important:**
This cell creates the real-world document corpus used by the venomous snake RAG
chatbot. The quality of chatbot responses depends on the quality and relevance
of the documents stored in `./my_data/`.

Using the Wikipedia API makes data collection reproducible and easier to cite
than manual copying. The selected species represent different regions of the
world: Africa, Asia, Australia, Central/South America, and South Asia/Middle
East. Their articles provide information on taxonomy, habitat, geographic
distribution, venom, behavior, conservation, and ecological importance.

The downloaded files are temporary because Google Colab storage resets when the
runtime disconnects. Therefore, this cell should be re-run whenever a new Colab
session starts.


---

In [1]:
import requests
import os

os.makedirs("./my_data", exist_ok=True)

# Wikipedia requires a descriptive User-Agent or it will block/throttle requests
HEADERS = {
    "User-Agent": "ITCC508-RAG-Lab/1.0 (educational project; contact: ceasaredaurdo.villenaiii@my.jru.edu)"
}

snake_pages = {
    "black_mamba": "Black mamba",
    "king_cobra": "King cobra",
    "inland_taipan": "Inland taipan",
    "fer_de_lance": "Bothrops asper",
    "saw_scaled_viper": "Echis carinatus"
}

def get_wikipedia_content(title):
    """Fetch full plain-text extract of a Wikipedia article using the Action API."""
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": True,
        "format": "json",
        "titles": title,
        "redirects": 1  # follow redirects (important for scientific names)
    }
    response = requests.get(url, params=params, headers=HEADERS, timeout=10)
    response.raise_for_status()
    data = response.json()

    pages = data["query"]["pages"]
    for page_id, page_data in pages.items():
        if page_id == "-1" or "extract" not in page_data:
            return None, None
        return page_data.get("title"), page_data.get("extract")
    return None, None


for filename, page_title in snake_pages.items():
    try:
        resolved_title, content = get_wikipedia_content(page_title)

        if content is None or len(content.strip()) == 0:
            print(f"❌ No content found for: {page_title}")
            continue

        filepath = f"./my_data/{filename}.txt"
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(f"Source: Wikipedia — {resolved_title}\n")
            f.write(f"URL: https://en.wikipedia.org/wiki/{resolved_title.replace(' ', '_')}\n")
            f.write("Licensed under CC BY-SA 4.0. Retrieved for academic use, ITCC508 Lab PT-M1.\n")
            f.write("=" * 70 + "\n\n")
            f.write(content)

        print(f"✅ Saved: {filepath} ({len(content)} characters)")

    except Exception as e:
        print(f"❌ Failed for {page_title}: {e}")

✅ Saved: ./my_data/black_mamba.txt (20296 characters)
✅ Saved: ./my_data/king_cobra.txt (19683 characters)
✅ Saved: ./my_data/inland_taipan.txt (26254 characters)
✅ Saved: ./my_data/fer_de_lance.txt (17085 characters)
✅ Saved: ./my_data/saw_scaled_viper.txt (8578 characters)


---
### **Cell 1: Library Imports and Setup**

**Purpose:**
Prepares the environment. Installs every package the RAG pipeline depends on: the LangChain framework, the Groq chat model integration, the Hugging Face embedding integration, the Chroma vector database, and the PDF/document parsing backends.

**Input:**
- Google Colab Python environment (fresh runtime)
- Internet connection to PyPI
- No API key or data files required yet

**Output:**
- All packages installed and importable
- Torch/transformers pulled in as dependencies of the embedding stack
- A ready-to-use environment for Cells 2–7

---

**Line-by-line breakdown:**

- `!pip install -q ...` — The `!` prefix tells the notebook (IPython/Jupyter/Colab) to run the rest of the line as a **shell command** in the terminal instead of as Python code; `pip install` fetches the packages from PyPI into the current kernel/runtime; `-q` is "quiet" mode, which suppresses most of the download and progress output so the cell output stays short.
- `langchain` — Core framework: the base abstractions for documents, prompts, chains, and runnables (LCEL) that every other cell uses.
- `langchain-community` — Third-party integrations that are not part of core: document loaders (`DirectoryLoader`), vector stores (`Chroma`), and more.
- `langchain-groq` — Groq's LangChain integration; provides `ChatGroq`, the class used in Cell 5 to call Groq's very fast inference API.
- `langchain-huggingface` — Hugging Face integration; provides `HuggingFaceEmbeddings` (Cell 4) and its LLM wrappers without you having to manually wire up the transformers/torch stack.
- `chromadb` — The actual vector database engine that the `Chroma` wrapper in Cell 4 runs on top of.
- `pypdf` — Backend used by LangChain's PDF loader (`PyPDFLoader`) to read PDF files.
- `unstructured` — Backend used by `UnstructuredFileLoader`; handles PDFs, Word docs, HTML, emails, and more. This is what `DirectoryLoader` falls back to for many file types.

---

### **Why this step is important:**

Every later cell imports from these packages, so nothing works without it. Two caveats: the installs live in the **runtime**, not in the saved notebook, so in Colab you must re-run this cell after every session reset; and it is the slowest cell in the notebook because it pulls heavy dependencies (torch, via the embedding stack).

---

In [2]:
# Cell 1: Environment Setup & Package Installation

!pip install -q langchain langchain-community langchain-groq langchain-huggingface chromadb pypdf unstructured

---
### **Cell 2: Secure API Key Configuration**

**Purpose:**
Supplies the Groq API key to the program without ever writing it in plain text inside the notebook. The key is typed interactively, hidden as it is entered, and stored in an environment variable where `ChatGroq` looks for it automatically in Cell 5.

**Input:**
- Your personal Groq API key (from https://console.groq.com/keys)
- Interactive keyboard input from you

**Output:**
- `GROQ_API_KEY` set in `os.environ`
- No visible trace of the key in the notebook output or shell history

---

**Line-by-line breakdown:**

- `import os` — Imports the `os` module, which gives access to **environment variables** through `os.environ`; this is the standard way to hand secrets to libraries, and LangChain's `ChatGroq` automatically looks for `GROQ_API_KEY` there.
- `import getpass` — Imports `getpass`, a module that can read input from the user **without echoing it** to the screen and without storing it in the notebook output or shell history.
- `if "GROQ_API_KEY" not in os.environ:` — Checks whether the key is already set, which makes the cell **idempotent**: re-running it will not ask you again, and it will not overwrite a key you set another way (for example through Colab Secrets).
- `os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")` — `getpass.getpass(...)` prints the prompt, hides your typing, and returns what you typed; that string is then assigned to `os.environ["GROQ_API_KEY"]`, so Cell 5's `ChatGroq` can pick it up with **zero hard-coded secrets** in the notebook.

---

### **Why this step is important:**

If you instead wrote `os.environ["GROQ_API_KEY"] = "gsk_abc123..."`, the key would be baked into the `.ipynb` JSON and leaked the moment you shared or committed the file. A leaked key means someone else can spend your API quota. This pattern keeps the secret out of the saved notebook entirely.

---

In [3]:
# Cell 2: Secure API Key Configuration

import os
import getpass

# Prompts for API key securely without saving or echoing plain text in notebook cells

if "GROQ_API_KEY" not in os.environ:
  os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


---
### **Cell 3: Loading Custom Data, Metadata Separation & Chunking**

**Purpose:**
Loads all snake documents from `./my_data/`, separates the citation/attribution
header from the actual article text, splits the cleaned content into smaller
semantic chunks, and filters out any chunks that contain little to no real
information (such as bare section headers). This produces the final, clean
`documents` list used to build the vector database in Cell 4.

**Input:**
- Five `.txt` files inside `./my_data/`, each containing a Wikipedia
  attribution header followed by a `"="*70` separator and the full article text
- No API key required

**Output:**
- `raw_documents` — the loaded files, with attribution text moved into
  `doc.metadata["attribution"]` and removed from `doc.page_content`
- `documents` — the final, filtered list of chunks ready for embedding
- Console output showing how many chunks were created and how many were
  removed for having too little content

---
**Line-by-line breakdown:**

- `from langchain_community.document_loaders import DirectoryLoader` — Imports
the loader class that can read every file inside a folder automatically,
instead of opening each `.txt` file manually one by one.

- `from langchain_text_splitters import RecursiveCharacterTextSplitter` —
Imports the tool that breaks long documents into smaller, overlapping chunks
suitable for embedding and retrieval.

- `import os` — Used to create the data folder and confirm it exists.

- `import re` — Python's regular expression module, used later to detect and
strip Wikipedia-style section headers (e.g. `== Description ==`) when checking
whether a chunk has enough real content.

- `os.makedirs("my_data", exist_ok=True)` — Creates the `my_data` folder if it
does not already exist. `exist_ok=True` prevents an error if it is already
there, so the cell can be safely re-run.

- `loader = DirectoryLoader("my_data/", glob="**/*.*", show_progress=True)` —
Creates a loader pointed at the `my_data` folder. `glob="**/*.*"` tells it to
match every file in the folder (and any subfolders) regardless of extension.
`show_progress=True` displays a progress bar while files are read.

- `raw_documents = loader.load()` — Reads every matched file and converts each
one into a LangChain `Document` object, storing the text in `.page_content`
and basic info (like the file path) in `.metadata`.

**Step 1 — Separating attribution from real content**

- `SEPARATOR = "=" * 70` — Recreates the exact 70-character line
(`"======...==="`) that was written into every file in Cell 2, right after
the source/URL/license header and before the actual Wikipedia article text.
This string is used as a splitting marker.

- `for doc in raw_documents:` — Loops through every loaded document.

- `if SEPARATOR in doc.page_content:` — Checks whether this particular
document actually contains the separator line (a safety check in case a file
was formatted differently).

- `header_part, body_part = doc.page_content.split(SEPARATOR, 1)` — Splits the
document's text into exactly two pieces at the first occurrence of the
separator: everything before it (`header_part`, the citation block) and
everything after it (`body_part`, the real article).

- `doc.metadata["attribution"] = header_part.strip()` — Saves the citation
text into the document's metadata dictionary under a new key called
`"attribution"`. Metadata is **not** embedded or searched — it is only carried
along for reference, so this information can still be displayed for citation
purposes without ever influencing similarity search results.

- `doc.page_content = body_part.strip()` — Replaces the document's main text
with only the real article content, discarding the citation header from the
part that will actually be embedded. `.strip()` removes leading/trailing
blank lines left over from the split.

- `print("✅ Attribution headers separated...")` — Confirms this step ran.

**Why this step was necessary:** During testing, the citation header (containing
words like "Source," "Retrieved," and "documents") was being embedded as
regular content. Because it used administrative language similar to some user
questions (e.g. "summarize the documents"), it was sometimes retrieved
*instead of* real snake information, causing the chatbot to incorrectly reply
that no answer was found. Moving this text into `metadata` permanently
prevents it from competing in similarity search.

**Step 2 — Splitting into chunks**

- `text_splitter = RecursiveCharacterTextSplitter(...)` — Creates the object
that will break long article text into smaller pieces.

- `chunk_size=800` — Each chunk will contain at most roughly 800 characters.
This was increased from an earlier value of 500 to give section headers (like
`== Description ==`) more room to stay attached to the paragraph that follows
them, instead of being cut off into their own tiny chunk.

- `chunk_overlap=100` — Each chunk shares its last 100 characters with the
start of the next chunk. This prevents important sentences from being cut
exactly in half at a chunk boundary and losing meaning.

- `separators=["\n\n", "\n", ". ", " ", ""]` — Tells the splitter which
boundaries to prefer when deciding where to cut text, in order of priority:
first try to split at paragraph breaks (`\n\n`), then single line breaks,
then sentence endings, then spaces, and only as a last resort in the middle
of a word. This produces more natural, readable chunks than relying on the
default settings.

- `documents = text_splitter.split_documents(raw_documents)` — Runs the
splitter on all cleaned documents and returns the full list of chunks.

**Step 3 — Filtering out low-content chunks**

- `def is_low_content_chunk(text, min_words=6):` — Defines a helper function
that decides whether a chunk contains too little real information to be
useful.

- `cleaned = re.sub(r'==+.*?==+', '', text).strip()` — Uses a regular
expression to remove Wikipedia-style section headers such as
`== Description ==` or `=== Scalation ===` from the text before counting
words. Without this step, a chunk that is literally just a header would still
"look" like it has content, since the header itself contains words.

- `word_count = len(cleaned.split())` — Counts how many words remain after
removing the header markup.

- `return word_count < min_words` — Returns `True` (meaning "discard this
chunk") if fewer than 6 real words remain — this reliably catches chunks that
are empty, header-only, or otherwise too short to answer any question.

- `original_count = len(documents)` — Records how many chunks existed before
filtering, for reporting purposes.

- `documents = [doc for doc in documents if not is_low_content_chunk(doc.page_content)]`
— Rebuilds the `documents` list, keeping only chunks that passed the content
check.

- `filtered_count = len(documents)` — Records the final chunk count after
filtering.

- The three `print(...)` lines — Report how many raw documents were loaded,
how many chunks were created before filtering, how many were removed, and how
many remain. These numbers are useful evidence for the lab report and for
verifying the vector database is built from clean data in Cell 4.

---
### **Why this step is important:**
The quality of every chatbot answer depends entirely on the quality of the
chunks stored in the vector database. Two problems discovered during testing
— citation headers being treated as searchable content, and bare section
headers being indexed as if they were real information — both caused the
chatbot to incorrectly refuse to answer questions that were genuinely
supported by the source documents. Fixing both issues at the chunking stage,
before anything is embedded, ensures the retriever can only return chunks
that contain real, useful snake information.

---

In [4]:
# Cell 3: Loading Custom Data & Chunking
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import re

# Create target data directory
os.makedirs("my_data", exist_ok=True)

# Load all documents from the directory
loader = DirectoryLoader("my_data/", glob="**/*.*", show_progress=True)
raw_documents = loader.load()

# --- STEP 1: Separate the attribution header from the real article content ---
SEPARATOR = "=" * 70

for doc in raw_documents:
    if SEPARATOR in doc.page_content:
        header_part, body_part = doc.page_content.split(SEPARATOR, 1)
        doc.metadata["attribution"] = header_part.strip()   # kept for citation only
        doc.page_content = body_part.strip()                # only real article text remains
    # If no separator is found, leave the document untouched

print("✅ Attribution headers separated from embeddable content.")

# --- STEP 2: Split documents into smaller semantic chunks ---
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

documents = text_splitter.split_documents(raw_documents)

# --- STEP 3: Remove low-content chunks (headers with little/no real text) ---
def is_low_content_chunk(text, min_words=6):
    # Strip section header markers like "== Description ==" before counting words
    cleaned = re.sub(r'==+.*?==+', '', text).strip()
    word_count = len(cleaned.split())
    return word_count < min_words

original_count = len(documents)
documents = [doc for doc in documents if not is_low_content_chunk(doc.page_content)]
filtered_count = len(documents)

print(f"Loaded {len(raw_documents)} raw document(s) and split into {original_count} chunks.")
print(f"Removed {original_count - filtered_count} low-content chunk(s).")
print(f"Final chunk count used for indexing: {filtered_count}")

/tmp/ipykernel_67687/3563513548.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader
100%|██████████| 5/5 [00:05<00:00,  1.09s/it]

✅ Attribution headers separated from embeddable content.
Loaded 5 raw document(s) and split into 168 chunks.
Removed 11 low-content chunk(s).
Final chunk count used for indexing: 157


---
### **Cell 4: Embedding Model & Vector Database Indexing**

**Purpose:**
Converts each cleaned, filtered text chunk from Cell 3 into a numerical vector
(embedding) using a Hugging Face sentence-embedding model, and stores those
vectors in a Chroma vector database so they can later be searched by
similarity. Also creates the `retriever` object used by the RAG chain.

**Input:**
- `documents` — the final filtered list of chunks produced in Cell 3
- No API key required; the embedding model runs locally after downloading

**Output:**
- `embeddings` — the loaded embedding model
- `vectorstore` — a Chroma collection containing one vector per chunk
- `retriever` — a search interface that returns the top-k most similar chunks
  for any given question

---
**Line-by-line breakdown:**

- `from langchain_huggingface import HuggingFaceEmbeddings` — Imports the
class that wraps a Hugging Face embedding model so it can be used directly
inside LangChain.

- `from langchain_community.vectorstores import Chroma` — Imports the wrapper
around the ChromaDB vector database.

- `embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")` —
Downloads (on first run) and loads the `all-MiniLM-L6-v2` sentence-embedding
model. This model converts any piece of text into a fixed-length numeric
vector that represents its meaning, allowing semantically similar pieces of
text to be found later using distance calculations rather than exact keyword
matches.

- `vectorstore = Chroma.from_documents(...)` — Creates a new Chroma vector
database and immediately embeds and stores every chunk inside it.

- `documents=documents` — Passes in the cleaned, filtered chunks from Cell 3
to be embedded and indexed.

- `embedding=embeddings` — Tells Chroma which embedding model to use to
convert each chunk's text into a vector before storing it.

- `collection_name="venomwise_snakes_v3"` — Gives this vector database
collection an explicit, unique name. This was added after discovering that
re-running this cell multiple times without changing the collection name
caused old and new chunks to accumulate together in the same collection,
leading to outdated or broken chunks being retrieved even after Cell 3 was
fixed. Using a new name each time the chunking logic changes guarantees the
vector store is rebuilt completely from scratch.

- `print(f"✅ Vector store rebuilt with {vectorstore._collection.count()} chunks (documents variable has {len(documents)}).")`
— Prints how many vectors actually exist inside the Chroma collection, and
compares it to how many chunks were passed in. These two numbers should
always match; if they do not, it is a sign the vector store still contains
leftover data from a previous run.

- `retriever = vectorstore.as_retriever(search_kwargs={"k": 3})` — Converts
the vector store into a retriever object that can be plugged into a RAG
chain. `search_kwargs={"k": 3}` tells it to return the 3 chunks most similar
to any given question, regardless of how many total chunks exist in the
database.

---
### **Why this step is important:**
This cell builds the actual searchable "memory" of the chatbot. Every answer
the LLM produces is grounded only in whatever the retriever finds here, so the
accuracy of this stage directly determines the accuracy and reliability of
every downstream response. Explicitly naming the collection and printing the
chunk counts were both added specifically to guarantee that this memory only
ever reflects the current, cleaned version of the documents — never a stale
or duplicated version left over from earlier debugging runs.

---

In [5]:
# Cell 4: Embedding Model & Vector DB Indexing
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize open-source embedding mode
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store embeddings into Chroma vector database
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="venomwise_snakes_v3"   # incremented again — guarantees a clean rebuild
)

print(f"✅ Vector store rebuilt with {vectorstore._collection.count()} chunks (documents variable has {len(documents)}).")

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Vector store rebuilt with 157 chunks (documents variable has 157).


---
### **Cell 5: Model Initialization and Domain System Prompt**

**Purpose:**
Initializes the Groq-hosted language model that will generate answers, and
defines the system prompt that restricts it to only use retrieved snake
information, permits it to summarize and compare that information, and
instructs it to refuse politely when no relevant information is found.

**Input:**
- A valid `GROQ_API_KEY` already stored in `os.environ` from Cell 2
- No document data required directly in this cell

**Output:**
- `llm` — the configured Groq chat model object
- `system_prompt` — the instruction text that will be prepended to every
  request sent to the model
- `prompt` — the full chat prompt template combining the system instructions
  with the user's question

---
**Line-by-line breakdown:**

- `from langchain_groq import ChatGroq` — Imports the class that connects
LangChain to Groq's hosted inference API.

- `from langchain_core.prompts import ChatPromptTemplate` — Imports the class
used to build a structured, reusable chat prompt made of a system message and
a human message.

- `llm = ChatGroq(...)` — Creates the model object that will generate answers.

- `model_name="openai/gpt-oss-20b"` — Specifies which hosted model to use.
This was changed from the originally specified `llama-3.1-8b-instant`, which
returned a 404 "model not found" error, confirming it had been deprecated
and removed from Groq's available model list. `openai/gpt-oss-20b` was
selected as a comparably sized, instruction-tuned replacement after checking
the live list of models available to the API key via Groq's `/v1/models`
endpoint.

- `temperature=0` — Sets the model's randomness to the minimum. At
`temperature=0`, the model always chooses its most likely next word, making
answers as consistent and deterministic as possible. This is the baseline
configuration used for grounded, low-hallucination testing.

- `system_prompt = (...)` — Builds the instruction text that tells the model
how to behave, using a multi-line string.

- `"You are VenomWise, a specialized AI assistant for venomous snake education.\n"`
— Gives the model a persona and clearly states the chatbot's domain.

- `"Answer questions strictly using ONLY the provided context below."` —
The core grounding rule: the model must not use any outside knowledge, only
the retrieved chunks supplied with each request.

- `"The terms 'context' and 'documents' refer to the same retrieved material.\n"`
— Added after testing revealed the model sometimes treated "documents" and
"context" as two different things, causing it to incorrectly refuse
legitimate summarization requests. This line removes that ambiguity.

- `"You are allowed and encouraged to summarize, compare, and explain retrieved information.\n"`
— Added to explicitly permit higher-level reasoning tasks (like comparing two
snakes), not just direct fact lookup. Without this line, the model was overly
cautious and refused reasonable requests even when the necessary information
was present in the retrieved chunks.

- `"If the answer cannot be found in the context, reply exactly: 'I cannot answer based on the provided domain data.'\n\n"`
— Defines the required fallback behavior, which is directly tested in Task 2.

- `"Context:\n{context}"` — A placeholder that LangChain automatically fills
in with the actual retrieved chunk text every time a question is asked.

- `prompt = ChatPromptTemplate.from_messages([...])` — Combines the system
prompt and a placeholder for the user's question into one reusable template.

- `("system", system_prompt)` — Registers the instructions above as the
model's system-level behavior.

- `("human", "{input}")` — A placeholder that will be replaced with whatever
question the user actually asks.

---
### **Why this step is important:**
This cell defines both **what model answers the questions** and **the rules it
must follow while answering**. The model substitution documents a real
provider-side change (model deprecation), and the two added prompt lines
directly fix a false-refusal problem discovered during testing, where the
model rejected valid, answerable questions due to overly rigid or ambiguous
wording. Together, these changes balance the two competing goals of this lab:
minimizing hallucination while still allowing the chatbot to be genuinely
useful.

---

In [51]:
# Cell 5: Model Initialization and Domain System Prompt
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Initialize the SLM
llm = ChatGroq(
    model_name="openai/gpt-oss-20b",
    temperature=0
    )

# Custom domain system prompt
system_prompt = (
    "You are VenomWise, a specialized AI assistant for venomous snake education.\n"
    "Answer questions strictly using ONLY the provided context below. "
    "The terms 'context' and 'documents' refer to the same retrieved material.\n"
    "You are allowed and encouraged to summarize, compare, and explain retrieved information.\n"
    "If the answer cannot be found in the context, reply exactly: 'I cannot answer based on the provided domain data.'\n\n"
    "Context:\n{context}"
    )

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
     ("human", "{input}"),
    ])

---
### **Cell 6: Pipeline Assembly**

**Purpose:**
Combines the retriever (Cell 4) and the LLM with its prompt (Cell 5) into a
single, runnable RAG chain that automatically retrieves relevant chunks,
inserts them into the prompt, and generates a grounded answer.

**Input:**
- `retriever` from Cell 4
- `llm` and `prompt` from Cell 5

**Output:**
- `rag_chain` — the complete retrieval-augmented generation pipeline used in
  Cell 7 and all later testing tasks

---
**Line-by-line breakdown:**

- `from langchain_classic.chains import create_retrieval_chain` — Imports the
function that assembles a full retrieval + generation pipeline. This import
path was changed from `langchain.chains` after LangChain's version 1.0 update
moved these helper functions out of the core package and into a separate
`langchain-classic` compatibility package.

- `from langchain_classic.chains.combine_documents import create_stuff_documents_chain`
— Imports the function that defines how retrieved chunks are combined
("stuffed") together into a single block of context before being sent to the
LLM. Same import-path update as above.

- `combine_docs_chain = create_stuff_documents_chain(llm, prompt)` — Builds
the part of the pipeline responsible for taking the retrieved chunks, inserting
them into the `{context}` placeholder of the prompt from Cell 5, and sending
the completed prompt to the LLM.

- `rag_chain = create_retrieval_chain(retriever, combine_docs_chain)` —
Builds the final, complete pipeline: given a question, it first calls
`retriever` to fetch the top-3 relevant chunks, then passes those chunks and
the question into `combine_docs_chain` to generate the final answer.

---
### **Why this step is important:**
This cell is where retrieval and generation are actually connected into one
working system. The import path had to be corrected here specifically because
of a breaking change introduced in a newer major version of LangChain that
was not present when the original lab template was written — a practical
example of how third-party library updates can affect a working pipeline even
when none of the surrounding logic has changed.

---

In [52]:
# Cell 6: Pipeline Assembly
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Combine prompt and LLM to process context
combine_docs_chain = create_stuff_documents_chain(llm, prompt)

# Assemble full retrieval-augmented generation chain
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

---
### **Cell 7: Testing Your Custom Domain Chatbot**

**Purpose:**
Runs the finished pipeline on a sample question and verifies two things at once: that the model answers from your documents, and **which** chunks it used to do so (traceability).

**Input:**
- `user_query` — a question about your uploaded data
- The assembled `rag_chain` from Cell 6

**Output:**
- The grounded answer printed to the screen
- A numbered list of the source files behind the retrieved chunks

---

**Line-by-line breakdown:**

- `user_query = "Summarize the key information found in the documents."` — The test question, stored in a variable so you can swap it out for any other question about your data.
- `response = rag_chain.invoke({"input": user_query})` — Executes the whole pipeline (embed → retrieve → prompt → generate) and returns a result dict with the keys `input`, `context` and `answer`; the key **must** be `"input"` so it matches the `{input}` placeholder defined in Cell 5.
- `print("--- DOMAIN QUERY ANSWER ---")` — Prints a section header so the output is readable.
- `print(response["answer"])` — Prints the model's grounded answer, i.e. the `answer` key of the returned dict.
- `print("\n--- RETRIEVED SOURCE CHUNKS ---")` — The `\n` emits a leading blank line plus the header, separating the two sections of output.
- `for i, doc in enumerate(response["context"]):` — `response["context"]` is the list of `Document` objects the retriever actually found (up to `k=3`, set in Cell 4); `enumerate` gives you both the index `i` and the document `doc`, so you can number the chunks as you print them.
- `print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))` — `i+1` converts the 0-based Python index into human-friendly 1-based numbering, and `doc.metadata.get("source", "Unknown")` safely reads the source file path out of the metadata (the same metadata `RecursiveCharacterTextSplitter` propagated in Cell 3), falling back to `"Unknown"` if the key is missing. This is your **traceability/verification** step: it shows which files the answer was built from. Print `doc.page_content` as well if you want to inspect the retrieved text itself.

---

### **Why this step is important:**

This is where you confirm the guardrail actually holds. Run two tests: an **in-domain** question (should be answered from your files) and an **out-of-domain** one such as `"Who won the 1998 FIFA World Cup?"` (should return `I cannot answer based on the provided domain data.`). If the out-of-domain question gets a real answer, the chunks did not contain the information and the model ignored the system prompt — re-check the Cell 5 wording.

---

### **Quick reference: the whole pipeline at a glance**

| Cell | Stage | Key object produced |
|---|---|---|
| 1 | Install dependencies | importable packages |
| 2 | Secure the API key | `os.environ["GROQ_API_KEY"]` |
| 3 | Load & chunk documents | `documents` (list of chunks) |
| 4 | Embed & index | `vectorstore`, `retriever` |
| 5 | Model + domain prompt | `llm`, `prompt` |
| 6 | Assemble RAG chain | `rag_chain` |
| 7 | Query & verify | `response["answer"]` |

**Common fixes:** `0 raw document(s)` → upload files into `my_data/`; slow re-indexing → add `persist_directory="./chroma_db"` in Cell 4; vague answers → raise `k` to 5–6 or `chunk_size` to ~1000; model 404 → Groq deprecates model names, swap in a current one such as `llama-3.3-70b-versatile`; no memory between questions → wrap with `create_history_aware_retriever` and `MessagesPlaceholder("chat_history")`.

In [64]:
# Cell 7: Testing Your Custom Domain Chatbot

# Test Case 1: In-Domain Query
user_query = "Summarize the key information found in the documents."
response = rag_chain.invoke({"input": user_query})

print("--- DOMAIN QUERY ANSWER ---")
print(response["answer"])

print("\n--- RETRIEVED SOURCE CHUNKS ---")
for i, doc in enumerate(response["context"]):
  print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))

--- DOMAIN QUERY ANSWER ---
I’m sorry, but the information you’re looking for isn’t included in the provided context.

--- RETRIEVED SOURCE CHUNKS ---
Chunk 1 Source: my_data/king_cobra.txt
Chunk 2 Source: my_data/saw_scaled_viper.txt
Chunk 3 Source: my_data/black_mamba.txt
